In [ ]:
!pip install requests beautifulsoup4 openpyxl pandas -q

In [ ]:
# ============================================================
#   BUSCADOR MASIVO DE DNI - PERÚ (v6 - DEFINITIVO)
#   Usa nombres_vars + multipart/form-data + parsea data.message
# ============================================================

# !pip install requests beautifulsoup4 openpyxl pandas -q

import requests
import re
import json
import pandas as pd
import time
import random
from google.colab import files

# ============================================================
# PASO 1: Subir Excel
# ============================================================
print("📂 Sube tu archivo Excel con la columna 'DNI'")
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

df = pd.read_excel(nombre_archivo, dtype={"DNI": str})
df["DNI"] = df["DNI"].str.strip().str.zfill(8)
print(f"✅ Cargado: {nombre_archivo} | Total DNIs: {len(df)}")

# ============================================================
# PASO 2: Obtener tokens frescos (nombres_vars)
# ============================================================
BASE_URL = "https://dniperu.com/buscar-dni-nombres-apellidos/"
AJAX_URL = "https://dniperu.com/wp-admin/admin-ajax.php"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-PE,es;q=0.9",
    "Origin": "https://dniperu.com",
    "Referer": BASE_URL,
}

def obtener_tokens(session):
    r = session.get(BASE_URL, headers=HEADERS, timeout=15)
    # Usar nombres_vars (el correcto para este formulario)
    m = re.search(r'var\s+nombres_vars\s*=\s*(\{[^}]+\})', r.text)
    if not m:
        raise ValueError("No se encontró nombres_vars en la página")
    datos = json.loads(m.group(1))
    return datos["nonce"], datos["ephemeral_token"], datos["ephemeral_sig"]

# ============================================================
# PASO 3: Función de búsqueda
# ============================================================
def buscar_dni(session, dni: str, nonce: str, token: str, sig: str):
    # Usar multipart/form-data igual que el navegador (FormData)
    files_payload = {
        "action":   (None, "buscar_nombres"),
        "security": (None, nonce),
        "cc_token": (None, token),
        "cc_sig":   (None, sig),
        "dni4":     (None, dni),
        "company":  (None, ""),
    }
    try:
        r = session.post(AJAX_URL, files=files_payload, headers=HEADERS, timeout=15)
        data = r.json()

        msg = data.get("data", {}).get("message", "")

        if not data.get("success"):
            if "vencida" in msg.lower() or "inválida" in msg.lower() or "invalida" in msg.lower():
                return "RENOVAR_TOKEN", "", ""
            return "NO ENCONTRADO", "", ""

        # Parsear el texto plano del message:
        # Numero de DNI: 47582719
        # Nombres: SERGIO LUIS
        # Apellido Paterno: SAAVEDRA
        # Apellido Materno: FARFAN
        # Codigo de Verificacion: 3
        nombres = apellido_p = apellido_m = ""
        for linea in msg.splitlines():
            linea = linea.strip()
            if linea.startswith("Nombres:"):
                nombres = linea.split(":", 1)[-1].strip()
            elif linea.startswith("Apellido Paterno:"):
                apellido_p = linea.split(":", 1)[-1].strip()
            elif linea.startswith("Apellido Materno:"):
                apellido_m = linea.split(":", 1)[-1].strip()

        if nombres:
            return nombres, apellido_p, apellido_m

        return "NO ENCONTRADO", "", ""

    except Exception as e:
        return f"ERROR: {str(e)[:50]}", "", ""

# ============================================================
# PASO 4: Prueba rápida
# ============================================================
print("\n🔍 Probando con DNI de prueba: 47582719")
session = requests.Session()
nonce, token, sig = obtener_tokens(session)
print(f"   nonce: {nonce} | token: {token[:10]}...")
n, p, m = buscar_dni(session, "47582719", nonce, token, sig)
print(f"   Resultado: '{n}' | '{p}' | '{m}'")

if n.startswith("ERROR") or n in ("NO ENCONTRADO", "RENOVAR_TOKEN"):
    print("⚠️  Prueba fallida — revisa conexión antes de continuar.")
    raise SystemExit("Detención preventiva.")
else:
    print("✅ Prueba exitosa. Iniciando búsqueda masiva...\n")

# ============================================================
# PASO 5: Procesar todos los DNIs
# ============================================================
resultados_nombres    = []
resultados_ap_paterno = []
resultados_ap_materno = []
total   = len(df)
errores = 0

for i, dni in enumerate(df["DNI"], start=1):
    print(f"[{i:>4}/{total}] DNI: {dni} ...", end=" ", flush=True)

    nombres, ap_p, ap_m = buscar_dni(session, str(dni), nonce, token, sig)

    # Renovar si el token venció y reintentar
    if nombres == "RENOVAR_TOKEN":
        print("🔄 Renovando tokens...", end=" ", flush=True)
        session = requests.Session()
        nonce, token, sig = obtener_tokens(session)
        time.sleep(2)
        nombres, ap_p, ap_m = buscar_dni(session, str(dni), nonce, token, sig)

    resultados_nombres.append(nombres)
    resultados_ap_paterno.append(ap_p)
    resultados_ap_materno.append(ap_m)

    if nombres in ("NO ENCONTRADO",) or nombres.startswith("ERROR"):
        print(f"❌  {nombres}")
        errores += 1
    else:
        print(f"✅  {nombres} | {ap_p} | {ap_m}")

    # Renovar tokens cada 8 búsquedas (tokens expiran rápido)
    if i % 8 == 0:
        session = requests.Session()
        nonce, token, sig = obtener_tokens(session)
        print(f"   🔑 Tokens renovados")

    time.sleep(random.uniform(2.0, 3.5))

print(f"\n📊 Total: {total} | ✅ Encontrados: {total-errores} | ❌ Fallos: {errores}")

# ============================================================
# PASO 6: Exportar Excel con formato
# ============================================================
df["NOMBRES"]          = resultados_nombres
df["APELLIDO_PATERNO"] = resultados_ap_paterno
df["APELLIDO_MATERNO"] = resultados_ap_materno

nombre_salida = "resultado_dni_" + nombre_archivo
df.to_excel(nombre_salida, index=False)

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

wb = load_workbook(nombre_salida)
ws = wb.active

header_fill = PatternFill("solid", start_color="1F4E79")
header_font = Font(bold=True, color="FFFFFF", name="Arial", size=11)
for cell in ws[1]:
    cell.fill      = header_fill
    cell.font      = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

red_fill    = PatternFill("solid", start_color="FFD7D7")
orange_fill = PatternFill("solid", start_color="FFE5B4")
col_idx     = list(df.columns).index("NOMBRES")

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    val = str(row[col_idx].value or "")
    if val == "NO ENCONTRADO":
        for cell in row:
            cell.fill = red_fill
    elif val.startswith("ERROR"):
        for cell in row:
            cell.fill = orange_fill

for col in ws.columns:
    max_len = max((len(str(c.value or "")) for c in col), default=10)
    ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 45)

wb.save(nombre_salida)
print(f"✅ Excel guardado: {nombre_salida}")

files.download(nombre_salida)
print("📥 Descarga iniciada.")

📂 Sube tu archivo Excel con la columna 'DNI'


Saving 50 EFRA.xlsx to 50 EFRA.xlsx
✅ Cargado: 50 EFRA.xlsx | Total DNIs: 50

🔍 Probando con DNI de prueba: 47582719
   nonce: 7187441458 | token: b37c09e759...
   Resultado: 'SERGIO LUIS' | 'SAAVEDRA' | 'FARFAN'
✅ Prueba exitosa. Iniciando búsqueda masiva...

[   1/50] DNI: 47582719 ... ✅  SERGIO LUIS | SAAVEDRA | FARFAN
[   2/50] DNI: 17956869 ... ✅  SANTOS REYNALDO | SILVESTRE | ALVAREZ
[   3/50] DNI: 60406129 ... ✅  MAYCOL RICARDO | CRUZADO | CONTRERAS


KeyboardInterrupt: 